# 1 — ERM baselines on PACS

Domain generalisation on **PACS**: four domains rendered in different visual
styles (`art_painting`, `cartoon`, `photo`, `sketch`), seven shared classes
(dog, elephant, giraffe, guitar, horse, house, person), 9,991 images. The
task is to train on some domains and classify images from a domain never
seen during training.

This notebook establishes the baselines the rest of the repository is
measured against, in increasing order of what the model is allowed to use:

1. a pretrained backbone with an untrained head — the chance level;
2. a frozen backbone with a linear probe, one domain at a time;
3. a full fine-tune on one domain, evaluated on the other three;
4. a fine-tune on three domains, evaluated on the fourth (leave-one-out);
5. the same, with photometric augmentation on the source domains;
6. train-one/evaluate-others, with augmentation.

`ResNet18` pretrained on ImageNet throughout, SGD with momentum 0.9,
lr 1e-3, batch 32, four epochs, cross-entropy.

The data is not committed. Lay PACS out as `<root>/<domain>/<class>/*.jpg`
and set the `PACS_ROOT` environment variable to `<root>`; it defaults to
`PACS/kfold` relative to the working directory. See the README.

## 1.1 Dataset

Per-class counts per domain, and three images from each domain. The class
totals differ sharply between domains — sketch has 816 horses and 80
houses, photo has 280 houses and 199 horses — so a per-domain majority-class
classifier is a stronger baseline than 1/7.

In [ ]:
import os
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from collections import Counter
pacs_root = os.environ.get("PACS_ROOT", os.path.join("PACS", "kfold"))
domains = ['art_painting', 'cartoon', 'photo', 'sketch']
num_images_to_show = 3
simple_transform = transforms.ToTensor()
fig, axs = plt.subplots(len(domains), num_images_to_show, figsize=(num_images_to_show * 3, len(domains) * 3))
for i, domain in enumerate(domains):
    domain_path = os.path.join(pacs_root, domain)
    dataset = datasets.ImageFolder(domain_path, transform=simple_transform)
    label_counts = Counter([label for _, label in dataset])
    class_names = dataset.classes
    print(f"\n{domain.upper()} DOMAIN")
    print(f"- {len(class_names)} unique labels")
    for class_id, count in sorted(label_counts.items()):
        print(f"  {class_names[class_id]}: {count} samples")
    for j in range(num_images_to_show):
        img, label = dataset[j]
        ax = axs[i, j] if len(domains) > 1 else axs[j]
        ax.imshow(img.permute(1, 2, 0))
        ax.set_title(f"{domain}\n{class_names[label]}")
        ax.axis('off')
plt.tight_layout()
plt.show()

## 1.2 Chance level

No training at all: ImageNet weights, a fresh 7-way head.

In [ ]:
# The pretrained backbone is kept but `fc` is replaced by a freshly
# initialised 7-way layer and never trained, so what follows is the chance
# level of the protocol (1/7 = 14.3% under a uniform prior), not a transfer
# result. The spread across domains comes from the class imbalance in each
# one interacting with whichever logit the random head happens to prefer.
import os
import torch
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
pacs_root = os.environ.get("PACS_ROOT", os.path.join("PACS", "kfold"))
domains = ['art_painting', 'cartoon', 'photo', 'sketch']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225])
])
base_model = models.resnet18(pretrained=True)
num_features = base_model.fc.in_features
num_pacs_classes = 7
base_model.fc = torch.nn.Linear(num_features, num_pacs_classes)
base_model = base_model.to(device)
base_model.eval()
def evaluate_domain(domain_path):
    dataset = datasets.ImageFolder(domain_path, transform=transform)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    total = 0
    correct = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Evaluating {os.path.basename(domain_path)}"):
            images, labels = images.to(device), labels.to(device)
            outputs = base_model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = 100.0 * correct / total
    print(f"[{os.path.basename(domain_path)}] Untrained-head accuracy: {acc:.2f}% ({correct}/{total})")
for domain in domains:
    domain_path = os.path.join(pacs_root, domain)
    evaluate_domain(domain_path)

## 1.3 Frozen backbone, linear probe per domain

Only `fc` is trained. Each domain is fitted and scored on itself, so the
accuracies below are **training** accuracies — see the comment in the cell.

In [ ]:
# Linear probe: the backbone is frozen and only `fc` is trained, on the whole
# of one domain. The evaluation loader below wraps `train_dataset` -- the same
# images the head was just fitted on -- so the number reported is training
# accuracy. It is kept because the report quotes it, but it is not a
# generalisation measurement and is not comparable to anything in notebook 03.
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
pacs_root = os.environ.get("PACS_ROOT", os.path.join("PACS", "kfold"))
domains = ['art_painting', 'cartoon', 'photo', 'sketch']
num_classes = 7
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])
for domain in domains:
    print(f"\n=== Fine-tuning on domain: {domain} ===")
    base_model = models.resnet18(pretrained=True)
    for param in base_model.parameters():
        param.requires_grad = False
    num_features = base_model.fc.in_features
    base_model.fc = nn.Linear(num_features, num_classes)
    base_model = base_model.to(device)
    optimizer = optim.SGD(base_model.fc.parameters(), lr=0.001, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    train_path = os.path.join(pacs_root, domain)
    train_dataset = datasets.ImageFolder(train_path, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    base_model.train()
    for epoch in range(4):
        total_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Training {domain} | Epoch {epoch+1}/4"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = base_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")
    base_model.eval()
    train_eval_loader = DataLoader(train_dataset, batch_size=32, shuffle=False, num_workers=0)
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in tqdm(train_eval_loader, desc=f"Evaluating on {domain}"):
            x, y = x.to(device), y.to(device)
            preds = base_model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    acc = 100 * correct / total
    print(f"[{domain}] Train-set accuracy: {acc:.2f}% ({correct}/{total})")

## 1.4 Fine-tune on one domain, evaluate on the other three

The whole network is trained on a single domain and then evaluated on each
of the other three. This is the pairwise transfer matrix: it shows which
style gaps are cheap to cross (photo → art_painting) and which are not
(photo → cartoon, sketch → photo).

In [ ]:
# Full fine-tune on one domain, evaluated on the other three. `evaluate_model`
# is only called for `d != domain_name`, so the train-domain diagonal of the
# resulting table has to come from the linear-probe cell above -- a different
# training regime, and a training-set number. The two are not comparable; see
# docs/report.md. (`pacs_root` is defined here; the original cell relied on
# the binding left behind by the cell above it.)
import os
pacs_root = os.environ.get("PACS_ROOT", os.path.join("PACS", "kfold"))

import os
import torch
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm
num_classes = 7
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
def evaluate_model(model, domain, transform):
    domain_path = os.path.join(pacs_root, domain)
    dataset = datasets.ImageFolder(domain_path, transform=transform)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Evaluating {domain}"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    accuracy = 100.0 * correct / total
    return accuracy
def fine_tune_resnet(domain_name):
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    train_domain_path = os.path.join(pacs_root, domain_name)
    train_dataset = datasets.ImageFolder(train_domain_path, transform=train_transform)
    train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    num_epochs = 4
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, labels in tqdm(train_dataloader, desc=f"Training on {domain_name}, Epoch {epoch+1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss / len(train_dataloader):.4f}")
    accuracies = {}
    for d in domains:
        if d != domain_name:
            accuracy = evaluate_model(model, d, eval_transform)
            accuracies[d] = accuracy
            print(f"Accuracy on {d} after fine-tuning on {domain_name}: {accuracy:.2f}%")
    return accuracies
for domain in domains:
    print(f"Fine-tuning on {domain} domain:")
    accuracies = fine_tune_resnet(domain)
    print("\n")

## 1.5 Leave-one-domain-out

Three domains for training, the fourth held out entirely.

In [ ]:
# Leave-one-domain-out, but note the loop structure: for each epoch the model
# sees one full pass over cartoon, then one full pass over photo, then one
# full pass over sketch. Gradients are never mixed across domains within a
# batch, so the model is repeatedly pulled toward whichever domain it saw
# last, and each epoch ends biased toward the final loader. This is
# sequential fine-tuning, not empirical risk minimisation over the pooled
# source distribution, and it is the main reason the accuracies below sit far
# under the published PACS baselines. Notebook 03 draws one batch from every
# source domain and concatenates them, which is the standard protocol.
# The behaviour is left in place because the report's numbers come from it.
import os
import torch
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm
pacs_root = os.environ.get("PACS_ROOT", os.path.join("PACS", "kfold"))
domains = ['art_painting', 'cartoon', 'photo', 'sketch']
num_classes = 7
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
def evaluate_model(model, domain, transform):
    domain_path = os.path.join(pacs_root, domain)
    dataset = datasets.ImageFolder(domain_path, transform=transform)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Evaluating {domain}"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    accuracy = 100.0 * correct / total
    return accuracy
def fine_tune_resnet_on_three_and_evaluate_on_fourth(training_domains, evaluation_domain):
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    train_datasets = [datasets.ImageFolder(os.path.join(pacs_root, domain), transform=train_transform) for domain in training_domains]
    train_dataloaders = [DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0) for train_dataset in train_datasets]
    num_epochs = 4
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        num_batches = 0
        for dataloader in train_dataloaders:
            for images, labels in tqdm(dataloader, desc=f"Training on {', '.join(training_domains)}, Epoch {epoch+1}/{num_epochs}"):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                num_batches += 1
        # The original divided the three-domain loss sum by len(dataloader), the
        # length of whichever loader the loop happened to exit on.
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss / num_batches:.4f}")
    accuracy = evaluate_model(model, evaluation_domain, eval_transform)
    print(f"Accuracy on {evaluation_domain} after fine-tuning on {', '.join(training_domains)}: {accuracy:.2f}%")
for i in range(len(domains)):
    training_domains = [domains[j] for j in range(len(domains)) if j != i]
    evaluation_domain = domains[i]
    print(f"\nFine-tuning on domains: {', '.join(training_domains)} and evaluating on domain: {evaluation_domain}")
    fine_tune_resnet_on_three_and_evaluate_on_fourth(training_domains, evaluation_domain)

## 1.6 Leave-one-domain-out with augmentation

Colour jitter, Gaussian blur and additive Gaussian noise on the source
domains. The held-out domain is evaluated with the plain transform.

In [ ]:
# Same leave-one-domain-out loop as above, with photometric augmentation on the
# source domains only: colour jitter, Gaussian blur, additive Gaussian noise.
# (`set_seed` calls np.random.seed, and numpy was not imported in the original
# cell -- it only ran because an earlier cell had bound `np` in the kernel.)
import torchvision.transforms as transforms
import torch
import numpy as np
import random
import os
import torch
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pacs_root = os.environ.get("PACS_ROOT", os.path.join("PACS", "kfold"))
domains = ['art_painting', 'cartoon', 'photo', 'sketch']
num_classes = 7
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class AddGaussianNoise(object):
    def __init__(self, mean=0., std=0.01):
        self.mean = mean
        self.std = std
    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    AddGaussianNoise(0., 0.01),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
def evaluate_model(model, domain, transform):
    domain_path = os.path.join(pacs_root, domain)
    dataset = datasets.ImageFolder(domain_path, transform=transform)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Evaluating {domain}"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    accuracy = 100.0 * correct / total
    return accuracy
def fine_tune_resnet_on_three_and_evaluate_on_fourth(training_domains, evaluation_domain):
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    train_datasets = [datasets.ImageFolder(os.path.join(pacs_root, domain), transform=train_transform) for domain in training_domains]
    train_dataloaders = [DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0) for train_dataset in train_datasets]
    num_epochs = 4
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for dataloader in train_dataloaders:
            for images, labels in tqdm(dataloader, desc=f"Training on {', '.join(training_domains)}, Epoch {epoch+1}/{num_epochs}"):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss / len(dataloader)}")
    accuracy = evaluate_model(model, evaluation_domain, eval_transform)
    print(f"Accuracy on {evaluation_domain} after fine-tuning on {', '.join(training_domains)}: {accuracy:.2f}%")
for i in range(len(domains)):
    training_domains = [domains[j] for j in range(len(domains)) if j != i]
    evaluation_domain = domains[i]
    print(f"\nFine-tuning on domains: {', '.join(training_domains)} and evaluating on domain: {evaluation_domain}")
    fine_tune_resnet_on_three_and_evaluate_on_fourth(training_domains, evaluation_domain)

## 1.7 Train-one/evaluate-others with augmentation

The 1.4 matrix again, with the augmented source transform.

In [ ]:
# Train-one/evaluate-on-the-other-three again, now with the augmented source
# transform, to separate the effect of augmentation from the effect of
# pooling three domains.
import os
import torch
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm
pacs_root = os.environ.get("PACS_ROOT", os.path.join("PACS", "kfold"))
domains = ['art_painting', 'cartoon', 'photo', 'sketch']
num_classes = 7
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class AddGaussianNoise(object):
    def __init__(self, mean=0., std=0.01):
        self.mean = mean
        self.std = std
    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    AddGaussianNoise(0., 0.01),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
def evaluate_model(model, domain, transform):
    domain_path = os.path.join(pacs_root, domain)
    dataset = datasets.ImageFolder(domain_path, transform=transform)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Evaluating {domain}"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    accuracy = 100.0 * correct / total
    return accuracy
def fine_tune_resnet(domain_name):
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    train_domain_path = os.path.join(pacs_root, domain_name)
    train_dataset = datasets.ImageFolder(train_domain_path, transform=train_transform)
    train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    num_epochs = 4
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, labels in tqdm(train_dataloader, desc=f"Training on {domain_name}, Epoch {epoch+1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss / len(train_dataloader)}")
    accuracies = {}
    for d in domains:
        if d != domain_name:
            accuracy = evaluate_model(model, d, eval_transform)
            accuracies[d] = accuracy
            print(f"Accuracy on {d} after fine-tuning on {domain_name}: {accuracy:.2f}%")
    return accuracies
for domain in domains:
    print(f"Fine-tuning on {domain} domain:")
    accuracies = fine_tune_resnet(domain)
    print("\n")